# 📊 Job Market Analysis 2026

This notebook provides a comprehensive Exploratory Data Analysis (EDA) of the 2026 tech job market dataset (`job_market_2026.csv`).

### Objectives:
1. **Data Cleaning & Inspection**: Address missing values and format data types.
2. **Salary Insights**: Analyze compensation trends by job role, experience level, and location.
3. **Remote Work Trends**: Explore how remote work ratios impact compensation and availability.
4. **Skills In-Demand**: Identify the most sought-after technical skills in the 2026 job market.

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load dataset
df = pd.read_csv('job_market_2026.csv')
print(f"Dataset Loaded Successfully. Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Data Cleaning & Inspection

In [ ]:
# Summary information
print("--- Data Types & Non-Null Counts ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Clean missing values
# Impute missing numerical values (salary) with median per experience level
df['salary_inr'] = df.groupby('experience_level')['salary_inr'].transform(lambda x: x.fillna(x.median()))

# Impute missing categorical values
df['company'] = df['company'].fillna('Unknown')
df['skills'] = df['skills'].fillna('Not Specified')

# Convert posted_date to datetime
df['posted_date'] = pd.to_datetime(df['posted_date'])
df['posted_month'] = df['posted_date'].dt.strftime('%B')

print("Missing values after cleaning:")
print(df.isnull().sum().sum())
df.describe()

## 3. Exploratory Data Analysis (EDA)

### A. Salary Distribution by Experience Level

In [ ]:
plt.figure(figsize=(12, 6))
exp_order = ['Entry', 'Mid', 'Senior', 'Lead']
sns.boxplot(data=df, x='experience_level', y='salary_inr', order=exp_order, hue='experience_level', palette='Set2', legend=False)
plt.title('Salary Distribution by Experience Level (INR)', fontsize=14, fontweight='bold')
plt.xlabel('Experience Level', fontsize=12)
plt.ylabel('Salary (INR)', fontsize=12)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f'{x/1e5:.1f}L'))
plt.tight_layout()
plt.show()

### B. Top Paying Job Roles

In [ ]:
avg_salary_role = df.groupby('job_title')['salary_inr'].agg(['mean', 'median', 'count']).sort_values(by='median', ascending=False)

plt.figure(figsize=(12, 6))
ax = sns.barplot(x=avg_salary_role['median'], y=avg_salary_role.index, hue=avg_salary_role.index, palette='crest', legend=False)
plt.title('Median Salary by Job Title (INR)', fontsize=14, fontweight='bold')
plt.xlabel('Median Salary (INR)', fontsize=12)
plt.ylabel('Job Title', fontsize=12)
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f'{x/1e5:.1f}L'))
plt.tight_layout()
plt.show()

avg_salary_role

### C. Remote Work Distribution & Salary Comparison

In [ ]:
df['remote_status'] = df['remote_ratio'].map({0: 'On-site (0%)', 50: 'Hybrid (50%)', 100: 'Fully Remote (100%)'})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Remote ratio counts
sns.countplot(data=df, x='remote_status', ax=axes[0], hue='remote_status', palette='viridis', legend=False)
axes[0].set_title('Job Counts by Work Arrangement', fontsize=14, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Postings')

# Salary by remote status
sns.boxplot(data=df, x='remote_status', y='salary_inr', ax=axes[1], hue='remote_status', palette='viridis', legend=False)
axes[1].set_title('Salary Breakdown by Work Arrangement', fontsize=14, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Salary (INR)')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f'{x/1e5:.1f}L'))

plt.tight_layout()
plt.show()

### D. Most In-Demand Skills Analysis

In [ ]:
from collections import Counter

# Explode skills column
all_skills = df['skills'].str.split(', ').explode()
skill_counts = Counter(all_skills)
top_skills_df = pd.DataFrame(skill_counts.most_common(15), columns=['Skill', 'Demand_Count'])

plt.figure(figsize=(12, 6))
sns.barplot(data=top_skills_df, x='Demand_Count', y='Skill', hue='Skill', palette='magma', legend=False)
plt.title('Top 15 Most In-Demand Skills (2026)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Job Postings', fontsize=12)
plt.ylabel('Skill', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Key Insights Summary & Conclusion

- **Highest Paying Roles**: AI Engineers and Lead Data Scientists command top median salaries across tech postings.
- **Experience Premium**: Moving from Entry to Senior/Lead levels yields a significant step up in compensation.
- **Skills Focus**: Python, SQL, Machine Learning, PyTorch, and Power BI are among the most frequently required tools.
- **Remote Flexibility**: Hybrid and Remote arrangements represent a large portion of tech job opportunities in 2026.